# Amodal Fine-tuning -- Full Pipeline (Mask + Appearance + Ket hop)

Chay toan bo tren 1 notebook Kaggle, tu dau den cuoi.

**Truoc khi chay:** Settings -> Accelerator -> GPU T4 x1.

## 1. Setup

In [ ]:
!rm -rf repo
!git clone https://github.com/YouttyLe-DSAI/LAOC2WAM-Learning-Amodal-Object-Completion-With-World-Action-Model.git repo
%cd repo

In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets
!pip uninstall -y -q torchao

In [ ]:
import os
os.environ["PYTHONPATH"] = "."
import torch
print("GPU available:", torch.cuda.is_available())

## 2. Nhanh Mask

In [ ]:
!PYTHONPATH=. python scripts/01b_prepare_synthetic.py --images data/raw --out data/cocoa \
    --download_samples --n_occluders_per_image 8

In [ ]:
!PYTHONPATH=. python scripts/04a_train_mask_model.py --config configs/config.yaml --epochs 100

In [ ]:
!PYTHONPATH=. python scripts/06_evaluate_mask.py --config configs/config.yaml \
    --checkpoint outputs/mask_model/best.pt

## 3. Nhanh Appearance (DreamBooth + Prior Preservation Loss)

In [ ]:
!git clone https://github.com/google/dreambooth.git third_party/dreambooth-dataset
!ls third_party/dreambooth-dataset/dataset/

In [ ]:
SUBJECT = "backpack"
!PYTHONPATH=. python scripts/02b_prepare_dreambooth_data.py \
    --dreambooth_dir third_party/dreambooth-dataset \
    --subject {SUBJECT} \
    --out data/train_ready

In [ ]:
!PYTHONPATH=. python scripts/04_train_lora.py --config configs/config.yaml

In [ ]:
!ls outputs/lora/final
!zip -rq lora_{SUBJECT}.zip outputs/lora/final
print(f"Da nen checkpoint: lora_{SUBJECT}.zip")

In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
pipe.unet = PeftModel.from_pretrained(pipe.unet, "outputs/lora/final")

demo_prompt = f"a photo of sks {SUBJECT} on a beach"
demo_image = pipe(demo_prompt, num_inference_steps=30).images[0]
demo_image.save("demo_output.png")
demo_image

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import glob

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

ref_paths = sorted(glob.glob("data/train_ready/instance_images/*.jpg"))
sims = []
for ref_path in ref_paths:
    ref_image = Image.open(ref_path).convert("RGB")
    inputs = clip_processor(images=[demo_image, ref_image], return_tensors="pt").to("cuda")
    with torch.no_grad():
        vision_out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        embeds = clip_model.visual_projection(vision_out.pooler_output)
    embeds = embeds / embeds.norm(dim=-1, keepdim=True)
    sims.append((embeds[0] @ embeds[1]).item())

clip_avg = sum(sims) / len(sims)
print(f"CLIP similarity trung binh voi {len(ref_paths)} anh goc: {clip_avg:.4f}")
print(f"Chi tiet: {[round(s,4) for s in sims]}")

## 4. Test tinh on dinh voi prompt khac

In [ ]:
generator = torch.Generator("cuda").manual_seed(42)
test_image = pipe(f"a photo of sks {SUBJECT}, plain background", num_inference_steps=30, generator=generator).images[0]
test_image

## 5. Tong hop ket qua

In [ ]:
print("===== TONG HOP KET QUA =====")
print(f"Subject: sks {SUBJECT}")
print(f"CLIP similarity trung binh: {clip_avg:.4f}")
print(f"File can tai ve: lora_{SUBJECT}.zip, demo_output.png")